In [ ]:
#first run pip install opencv-python mediapipe numpy in your terminal
import sys
!{sys.executable} -m pip install opencv-python mediapipe numpy
import cv2
import mediapipe as mp
import time
import os
import urllib.request
import math
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles

# ==========================================
# 1. SETUP & MODEL DOWNLOAD
# ==========================================
model_path = 'face_landmarker.task'
if not os.path.exists(model_path):
    print("Downloading Face Landmarker model...")
    url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
    urllib.request.urlretrieve(url, model_path)
    print("Download complete!")

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1
)
detector = vision.FaceLandmarker.create_from_options(options)

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def get_distance(p1, p2, width, height):
    """Converts normalized landmarks to pixels and calculates Euclidean distance."""
    x1, y1 = int(p1.x * width), int(p1.y * height)
    x2, y2 = int(p2.x * width), int(p2.y * height)
    return math.hypot(x2 - x1, y2 - y1)

# ==========================================
# 3. MAIN WEBCAM LOOP
# ==========================================
cap = cv2.VideoCapture(0)
print("Press 'q' in the video window to quit.")
last_timestamp_ms = 0

while True:
    success, frame = cap.read()
    if not success:
        break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    
    current_timestamp_ms = int(time.time() * 1000)
    if current_timestamp_ms <= last_timestamp_ms:
        current_timestamp_ms = last_timestamp_ms + 1
    last_timestamp_ms = current_timestamp_ms
    
    result = detector.detect_for_video(mp_image, current_timestamp_ms)

    # Default state
    emotion = "Neutral"

    if result.face_landmarks:
        for face_landmarks in result.face_landmarks:
            
            # --- A. FEATURE EXTRACTION ---
            # 1. Mouth Width (Smile)
            left_mouth = face_landmarks[61]
            right_mouth = face_landmarks[291]
            mouth_width = get_distance(left_mouth, right_mouth, w, h)
            
            # 2. Mouth Openness (Surprise/Talking)
            top_lip = face_landmarks[13]
            bottom_lip = face_landmarks[14]
            mouth_openness = get_distance(top_lip, bottom_lip, w, h)
            
            # 3. Eye Openness (Surprise/Squint)
            left_eye_top = face_landmarks[159]
            left_eye_bottom = face_landmarks[145]
            left_eye_openness = get_distance(left_eye_top, left_eye_bottom, w, h)
            
            # 4. Brow Drop (Anger/Scowl)
            left_inner_brow = face_landmarks[107]
            brow_to_eye_dist = get_distance(left_inner_brow, left_eye_top, w, h)

            # --- B. THE RULE ENGINE ---
            if mouth_openness > 25 and left_eye_openness > 12:
                emotion = "Surprised"
                
            # NEW RULE: If eyes are squinting AND mouth is tightly closed
            elif left_eye_openness < 9 and mouth_openness < 5:  
                emotion = "Angry"
                
            elif mouth_width > 60:       
                emotion = "Happy"
                
            else:
                emotion = "Neutral"

            # --- C. VISUAL OUTPUT ---
            # Draw circles on key points so you can visualize what is being tracked
            points_to_draw = [left_mouth, right_mouth, top_lip, bottom_lip, 
                              left_eye_top, left_eye_bottom, left_inner_brow]
            
            for point in points_to_draw:
                cx, cy = int(point.x * w), int(point.y * h)
                cv2.circle(frame, (cx, cy), 3, (0, 255, 255), -1) 

            # Display the live metrics
            cv2.putText(frame, f"Mouth Width: {int(mouth_width)}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.putText(frame, f"Mouth Open: {int(mouth_openness)}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.putText(frame, f"Eye Open: {int(left_eye_openness)}", (10, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            cv2.putText(frame, f"Brow-Eye Dist: {int(brow_to_eye_dist)}", (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            
            # Display the final Emotion Decision
            cv2.putText(frame, f"EMOTION: {emotion}", (10, 170), cv2.FONT_HERSHEY_DUPLEX, 1.2, (0, 255, 0), 2)
    cv2.imshow("Rule-Based Emotion Detector", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
    --------------------------------------- 0.5/40.2 MB 10.9 MB/s eta 0:00:04
    --------------------------------------- 0.8/40.2 MB 1.8 MB/s eta 0:00:22
    --------------------------------------- 0.8/40.2 MB 1.8 MB/s eta 0:00:22
   - -------------------------------------- 1.0/40.2 MB 1.4 MB/s eta 0:00:29
   - -------------------------------------- 1.3/40.2 MB 1.3 MB/s eta 0:00:31
   - -------------------------------------- 1.6/40.2 MB 1.2 MB/s eta 0:00:33
   - -------------------------------------- 1.6/40.2 MB 1.2 MB/s eta 0:00:33
   - -------------------------------------- 1.6/40.2 MB 1.2 MB/s eta 0:00:33
   - -------------------------------------- 1.8/40.2 MB 945.7 kB/s eta 0:00:41
   -- ------------------------------------- 2.1/40.2 MB 937.2 kB/s eta 0:00:41
   -- ------------------------------------- 2.1/40.2 MB 937.2 kB/s eta 0:00:41
   -- ------------------------------------- 2.4/40.2 MB 941.5 kB/s eta 0:00: